# Gradient boosting

Adnan Masood · University of South Florida

Compare a straightforward probability model with a more complex boosted-tree model for a rare financial outcome. Ranking asks whether higher-risk cases tend to appear first; probability accuracy asks whether predicted chances match what happens. A strong ranking score does not guarantee accurate probabilities or better lending decisions.

**Dataset:** The computer creates 3,000 fictional rows with 20 numeric inputs. About 8 in every 100 outcomes are positive, and 1% of labels are deliberately noisy. A stratified split keeps roughly that outcome mix in the 70% training group and 30% test group. Seed 21 makes the examples repeatable.

**Task:** Fit scaled logistic regression and histogram gradient boosting on the training rows. Then use both models to estimate chances for the same held-back test rows. Compare ROC-AUC for ranking and Brier score for probability error; these measures answer different questions.

**Compare:** Compare both models on the exact same held-back rows. ROC-AUC checks how often a randomly chosen positive case receives a higher score than a negative case. Brier score averages squared differences between each estimated chance and its 0-or-1 outcome; lower is better. Neither score alone chooses a lending action.

**Runtime:** This notebook uses scikit-learn. Boosting uses a learning rate of 0.05 and limits each tree to 15 leaves. Use the training rows to fit the models, a separate validation group to choose settings, and keep the final test rows for one last check.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
X, y = make_classification(n_samples=3000, n_features=20,
    n_informative=10, weights=[.92, .08], flip_y=.01, random_state=21)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=.30, stratify=y, random_state=21)
models = {
 "logit": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
 "boost": HistGradientBoostingClassifier(max_iter=100,
    learning_rate=.05, max_leaf_nodes=15, random_state=21)
}
for name, m in models.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    print(name, "AUC:", roc_auc_score(yte,p), "Brier:", brier_score_loss(yte,p))


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Increase learning rate

Compare AUC and Brier while keeping 100 boosting iterations.

Replace:
```python
learning_rate=.05
```
With:
```python
learning_rate=.10
```

### Reduce leaves per tree

Compare both scores against the unchanged logistic baseline.

Replace:
```python
max_leaf_nodes=15
```
With:
```python
max_leaf_nodes=7
```

### Increase class imbalance

Compare both models on the regenerated dataset and state its changed prevalence.

Replace:
```python
weights=[.92, .08]
```
With:
```python
weights=[.97, .03]
```



In [ ]:
# Record AUC and Brier for learning_rate=0.05 and learning_rate=0.10.
# Keep the logistic baseline and the train/test split unchanged.


## Break it

Change the positive outcome share from about 8 in 100 rows to about 3 in 100. Rerun both models with the same split seed and compare ROC-AUC and Brier score. Since the outcome mix changed, a Brier score can change even if ranking does not; do not use this single test set repeatedly to select settings.


In [ ]:
# Set weights=[0.97, 0.03] and record both models' AUC and Brier.
# Identify which conclusions require another validation split.


## What to submit

- A table comparing ROC-AUC and Brier score for both models on the same test rows, with lower Brier identified as better.
- One controlled change to learning rate or leaf count, with all other settings held fixed.
- A recommendation that says whether ranking and probability accuracy point to the same model, and what evidence would still be needed before a decision policy.

## Optional extension

A reliability plot can check whether predicted chances match observed frequencies. A decision cutoff needs stated costs for missed defaults, mistaken declines, and manual review. This notebook does not calculate those policy choices or measure production operating time.
